In [1]:
import numpy as np 
from pathlib import Path
from liffile import LifFile
import xarray as xr
import os
from bioio.writers import OmeTiffWriter
from bioio_base.types import PhysicalPixelSizes

In [2]:
# uses LifFile to load .lif "project" file and returns "lif" which contains every image and all metadata in the project
def load_project(input_file):
    
    input = Path(input_file)
    print(f"processing file {input}")
    lif = LifFile(str(input))

    return lif

In [3]:
# load specified image and convert to xarray. Converts image from whatever dimension microscope specified into ome-required dimensions (TZCYX)

def load_image(lif_img):
    xdata = lif_img.asxarray() # convert to xarray
    curr_dims = list(lif_img.dims) # get the images current dimensions 
    ome_arr = ['T', 'Z', 'C', 'Y', 'X'] # specify the ome-dimensions 

    missing_ele = (set(ome_arr)-set(curr_dims)) # get the missing elements 

    for dim in ome_arr:
        if dim not in xdata.dims:
            xdata = xdata.expand_dims({dim: 1})

    xdata = xdata.transpose(*ome_arr)

    return xdata 

In [4]:
# saves a single specified image from the .lif file 

def save_single_image(lif_project, idx, out_path):

    xdata = load_image(lif_project, idx) # run load image function and return data 
    data = xdata.values # just grab the image values from the xdata array for the image data - this is saved into OmeTiffWriter
    
    # gets the image dimensions and returns axis scale of the image
    def get_res(axis): 
        
        coords = xdata.coords[axis].values # given [X,Y, or Z] axis get the value from the coords tag 
        scale = round((coords[1]-coords[0]) * 1e6, 4) if len(coords) > 1 else 1.0 # get the image scale by subtracting coords[1] form coords[0] on the axis and converting to µm from m (default leica unit)
        return scale 

    px_sizes = PhysicalPixelSizes(get_res('Z'), get_res('Y'), get_res('X')) # using PhysicalPixelSizes funciton from bioio to be used for saving ometiff

    channels = [f"Ch_{i}" for i in range(xdata.shape[xdata.dims.index('C')])] # get names of channels specified by scope 

    tif_name = f"{xdata.name.replace(' ', '_')}.ome.tif" # set tiff name equivalent to image name specified by scope 
    tif_path = out_path / tif_name # set output path 

    OmeTiffWriter.save(
        data,
        tif_path,
        dim_order="TZCYX",          
        channel_names = channels,
        physical_pixel_sizes=px_sizes
    )

In [ ]:
# save multiple images - this is what is ported over to the .py file 

def save_images(in_path, antigen, out_path):

    cwd = os.getcwd() 
    full_out = Path(cwd) / out_path / antigen / 'images' # setting output to cwd/user inputted directory/images

    project = load_project(in_path) # load specified .lif file 

    for image in project.images: # looping through each image in the .lif file 
        try:
            xdata = load_image(image) # load image 
            data = xdata.values # storing image values 
            
            # function to get scale of image 
            def get_res(axis):
                
                coords = xdata.coords[axis].values # given [X,Y, or Z] axis get the value from the coords tag 
                scale = round((coords[1]-coords[0]) * 1e6, 4) if len(coords) > 1 else 1.0 # get the image scale by subtracting coords[1] form coords[0] on the axis and converting to µm from m (default leica unit)
                return scale

            px_sizes = PhysicalPixelSizes(get_res('Z'), get_res('Y'), get_res('X')) # using PhysicalPixelSizes funciton from bioio to be used for saving ometiff
            channels = [f"Ch_{i}" for i in range(xdata.shape[xdata.dims.index('C')])] # getting channel names from current image 

            dir_path = Path(full_out) / image.name 
            Path.mkdir(dir_path, parents = True, exist_ok = True) # make a directory for each individual images to handle saving multiple copies easier later 
            
            tif_name = f"{xdata.name.replace(' ', '_')}.ome.tif" # creating name of new ome.tif 
            tif_path = dir_path / tif_name # getting whole path for saving new ome.tif

            # OmeTiffWriter from bioio which makes xml for you - way easier than tiffile 
            OmeTiffWriter.save(
                data,
                tif_path,
                dim_order = "TZCYX",
                channel_names = channels,
                physical_pixel_sizes = px_sizes
            )
            print(f"saved: {tif_name} | shape: {xdata.shape}")
        
        except Exception as e:
            print(f"failed to process {image.name}: {e}") # returns error if new tif not saved for some reason 

In [8]:
in_path = "raw_data/CD11c_titer_ligM2.lif" # define .lif directory 
antigen = "CD11c_ligM2" # define current antigen working on (this is just the name of the subdir output file though)
out_path = "output2" # define output directory
save_images(in_path, antigen, out_path) 

processing file raw_data/CD11c_titer_ligM2.lif
(1, 1, 2, 966, 1296)
saved: 100_20x_0-0.ome.tif | shape: (1, 1, 2, 966, 1296)
(1, 1, 2, 966, 1296)
saved: 100_20x_1-0.ome.tif | shape: (1, 1, 2, 966, 1296)
(1, 1, 2, 966, 1296)
saved: 100_20x_2-0.ome.tif | shape: (1, 1, 2, 966, 1296)
(1, 1, 2, 966, 1296)
saved: 200_20x_0-0.ome.tif | shape: (1, 1, 2, 966, 1296)
(1, 1, 2, 966, 1296)
saved: 200_20x_1-0.ome.tif | shape: (1, 1, 2, 966, 1296)
(1, 1, 2, 966, 1296)
saved: 200_20x_2-0.ome.tif | shape: (1, 1, 2, 966, 1296)
(1, 1, 2, 966, 1296)
saved: 400_20x_0-0.ome.tif | shape: (1, 1, 2, 966, 1296)
(1, 1, 2, 966, 1296)
saved: 400_20x_1-0.ome.tif | shape: (1, 1, 2, 966, 1296)
(1, 1, 2, 966, 1296)
saved: 400_20x_2-0.ome.tif | shape: (1, 1, 2, 966, 1296)
(1, 1, 2, 966, 1296)
saved: 800_20x_0-0.ome.tif | shape: (1, 1, 2, 966, 1296)
(1, 1, 2, 966, 1296)
saved: 800_20x_1-0.ome.tif | shape: (1, 1, 2, 966, 1296)
(1, 1, 2, 966, 1296)
saved: 800_20x_2-0.ome.tif | shape: (1, 1, 2, 966, 1296)


In [ ]:
project = load_project("raw_data/CD11b_titer_LigM2.lif")[0]
out = Path('output2')
img_idx = 0
xdata = load_image(project, img_idx)
save_single_image(project, img_idx, out)